# Chapter 14. Validation, fairness, and accountability

*Companion notebook to* **Computational Criminology: Research Methods, Ethics, and Reproducible Practice with R and Python** *by Dr Fahad Hameed Khan.*

Run the setup cell once, then run the code cell. Everything uses synthetic data, so nothing here describes a real person. The R version of this chapter is in `code/r/ch14_fairness.R`.

In [ ]:
# Setup: fetch the repository, install what this chapter needs,
# and build the synthetic data. Safe to run more than once.
import os
if not os.path.exists('computational-criminology'):
    !git clone -q https://github.com/drfahadhameedkhan/computational-criminology.git
%cd -q computational-criminology
!pip install -q numpy pandas matplotlib scikit-learn
!python data/synthetic/make_synthetic_data.py

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# --- setup: train the Chapter 8 model, split the test set by group ---------
DATA = Path("data/synthetic/ml_table.csv")
df = pd.read_csv(DATA)
X = df[["deprivation", "prior_count", "density"]]
y = df["target"].values
g = df["group"].values
Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, g, test_size=0.4,
                                                random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
s = clf.predict_proba(Xte)[:, 1]
yA, sA = yte[gte == "A"], s[gte == "A"]
yB, sB = yte[gte == "B"], s[gte == "B"]

# --- book code (Chapter 14) ------------------------------------------------
# Python: fairness metrics by group
def group_rates(y, score, threshold=0.5):
    pred = score >= threshold
    tp = np.sum(pred & (y==1)); fp = np.sum(pred & (y==0))
    fn = np.sum(~pred & (y==1)); tn = np.sum(~pred & (y==0))
    return dict(FPR=fp/(fp+tn), FNR=fn/(fn+tp),
                precision=tp/(tp+fp) if tp+fp else 0)
print("Group A:", {k: round(v,2) for k,v in group_rates(yA, sA).items()})
print("Group B:", {k: round(v,2) for k,v in group_rates(yB, sB).items()})

## Exercises

- Coding task. Take the prediction model you built in Chapter 8 and compute confusion matrices separately for two groups defined by a demographic or area characteristic. Report which fairness criteria hold, which fail, and in which direction.
- Coding task. Construct a synthetic dataset in which two groups have different base rates of the outcome. Show numerically that you can equalise calibration or false positive rates across groups by adjusting thresholds, but not both, and present the trade-off as a plot.
- Write a model card for your Chapter 8 model: intended use, training data and its known biases, performance overall and by group, and the uses for which the model must not be employed. Write it so an affected member of the public could follow it.
- Draft the accountability conditions a police force should be required to meet before deploying a risk model: what must be documented, who reviews it, how an affected person is informed, and how they contest a decision. Compare your conditions against the high-risk requirements of the EU AI Act.